In [142]:
import os
import glob
import re
import pandas as pd

In [143]:
def generate_paired_manifest():
    base_dir = "birads_dataset"
    jpeg_dir = os.path.join(base_dir, "jpeg")
    output_csv = "birads_dataset/clean_pairs.csv"
    
    # 1. Map all available JPEGs
    print("Scanning for JPEG images...")
    all_jpegs = glob.glob(os.path.join(jpeg_dir, "**", "*.jpg"), recursive=True)
    uid_to_jpg = {os.path.basename(os.path.dirname(jpg)): jpg for jpg in all_jpegs}

    # 2. Load and Combine Metadata
    mass_csv = os.path.join(base_dir, "csv/mass_case_description_train_set.csv")
    calc_csv = os.path.join(base_dir, "csv/calc_case_description_train_set.csv")
    
    dfs = []
    if os.path.exists(mass_csv):
        # Fix column inconsistency
        m_df = pd.read_csv(mass_csv).rename(columns={'breast_density': 'breast density'})
        dfs.append(m_df)
    if os.path.exists(calc_csv):
        dfs.append(pd.read_csv(calc_csv))
        
    df = pd.concat(dfs, ignore_index=True)
    df.columns = df.columns.str.strip() 

    # 3. Match Metadata to JPEGs
    def get_jpg_path(dicom_path):
        if not isinstance(dicom_path, str): return None
        for uid, jpg_path in uid_to_jpg.items():
            if uid in dicom_path: return jpg_path
        return None

    df['jpg_path'] = df['image file path'].apply(get_jpg_path)
    df_clean = df.dropna(subset=['jpg_path'])

    # 4. FIX: Aggregate Abnormality Labels to Image Level FIRST
    # If an image has multiple abnormalities, take the max BI-RADS assessment
    img_level = df_clean.groupby(
        ['patient_id', 'left or right breast', 'image view', 'jpg_path']
    ).agg(
        assessment=('assessment', 'max') # Take the worst-case BI-RADS score
    ).reset_index()

    # 5. Pair CC and MLO views
    print("Pairing CC and MLO views for each patient...")
    paired = img_level.pivot_table(
        index=['patient_id', 'left or right breast'], 
        columns='image view', 
        values=['jpg_path', 'assessment'], 
        aggfunc='first'
    )
    
    # Flatten the multi-index columns created by pivot_table
    paired.columns = [f"{col[0]}_{col[1]}" for col in paired.columns]
    paired = paired.reset_index()

    # Drop any records that don't have BOTH a CC and an MLO view
    paired = paired.dropna(subset=['jpg_path_CC', 'jpg_path_MLO'])

    # 6. Format for the Spatial Alignment Pipeline
    final_df = paired[['jpg_path_CC', 'jpg_path_MLO', 'assessment_CC', 'left or right breast']].copy()
    final_df = final_df.rename(columns={
        'jpg_path_CC': 'cc_image_path', 
        'jpg_path_MLO': 'mlo_image_path',
        'assessment_CC': 'birads_label', # CC and MLO will have the same overall patient assessment
        'left or right breast': 'breast_side'
    })

    final_df['birads_label'] = final_df['birads_label'].astype(int)
    final_df.to_csv(output_csv, index=False)
    print(f"\nSuccess! Cleaned and aggregated manifest saved to {output_csv}")
    print(f"Total valid CC/MLO pairs ready for STN: {len(final_df)}")

In [144]:
import os

print("base_dir exists:", os.path.exists("birads_dataset"))
print("jpeg_dir exists:", os.path.exists("birads_dataset/jpeg"))
print("mass csv exists:", os.path.exists("birads_dataset/mass_case_description_train_set.csv"))
print("calc csv exists:", os.path.exists("birads_dataset/calc_case_description_train_set.csv"))

base_dir exists: True
jpeg_dir exists: True
mass csv exists: False
calc csv exists: False


In [ ]:
generate_paired_manifest()

Scanning for JPEG images...


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix
from sklearn.metrics import f1_score, cohen_kappa_score
import timm
from torch.utils.data import WeightedRandomSampler

In [ ]:
df = pd.read_csv("birads_dataset/clean_pairs.csv")

print(df.head())
print("Rows:", len(df))
print("Labels:", sorted(df["birads_label"].unique()))

                                       cc_image_path  \
0  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.3...   
1  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.8...   
2  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.4...   
3  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.2...   
4  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.4...   

                                      mlo_image_path  birads_label breast_side  
0  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.3...             4        LEFT  
1  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.2...             4        LEFT  
2  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.2...             3       RIGHT  
3  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.1...             4        LEFT  
4  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.1...             2        LEFT  
Rows: 1062
Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [ ]:
counts = df["birads_label"].value_counts()
valid_labels = counts[counts >= 2].index

df = df[df["birads_label"].isin(valid_labels)].copy()
df = df[df["birads_label"] != 0].copy()
df = df[df["birads_label"] != 1].copy()
print(df["birads_label"].value_counts().sort_index())

birads_label
2    138
3    140
4    507
5    184
Name: count, dtype: int64


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["birads_label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["birads_label"]
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 775
Val: 97
Test: 97


In [ ]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.5.1+cu121
12.1
True
1


In [ ]:
os.makedirs("birads_dataset/splits", exist_ok=True)

train_df.to_csv("birads_dataset/splits/train.csv", index=False)
val_df.to_csv("birads_dataset/splits/val.csv", index=False)
test_df.to_csv("birads_dataset/splits/test.csv", index=False)

In [ ]:
classes = sorted(df["birads_label"].unique())
label_map = {c: i for i, c in enumerate(classes)}
inv_label_map = {i: c for c, i in label_map.items()}

print("label_map:", label_map)

for split_df in [train_df, val_df, test_df]:
    split_df["ordinal_label"] = split_df["birads_label"].map(label_map)

label_map: {np.int64(2): 0, np.int64(3): 1, np.int64(4): 2, np.int64(5): 3}


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
class SingleViewDataset(Dataset):
    def __init__(self, dataframe, image_col, transform=None, label_col="ordinal_label"):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_col = image_col
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row[self.image_col]).convert("RGB")
        label = int(row[self.label_col])

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
class PairedDataset(Dataset):
    def __init__(self, dataframe, transform=None, label_col="ordinal_label"):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        cc_img = Image.open(row["cc_image_path"]).convert("RGB")
        mlo_img = Image.open(row["mlo_image_path"]).convert("RGB")
        label = int(row[self.label_col])

        if self.transform:
            cc_img = self.transform(cc_img)
            mlo_img = self.transform(mlo_img)

        return cc_img, mlo_img, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
class_counts = train_df["ordinal_label"].value_counts().sort_index()
print(class_counts)

class_weights = 1.0 / class_counts
sample_weights = train_df["ordinal_label"].map(class_weights).values
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

ordinal_label
0     74
1    110
2    112
3    405
4    147
Name: count, dtype: int64


In [ ]:
batch_size = 16

cc_train_ds = SingleViewDataset(train_df, image_col="cc_image_path", transform=train_transform)
cc_val_ds   = SingleViewDataset(val_df, image_col="cc_image_path", transform=val_transform)
cc_test_ds  = SingleViewDataset(test_df, image_col="cc_image_path", transform=val_transform)

mlo_train_ds = SingleViewDataset(train_df, image_col="mlo_image_path", transform=train_transform)
mlo_val_ds   = SingleViewDataset(val_df, image_col="mlo_image_path", transform=val_transform)
mlo_test_ds  = SingleViewDataset(test_df, image_col="mlo_image_path", transform=val_transform)

paired_test_ds = PairedDataset(test_df, transform=val_transform)

cc_train_loader = DataLoader(cc_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
cc_val_loader   = DataLoader(cc_val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
cc_test_loader  = DataLoader(cc_test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

mlo_train_loader = DataLoader(mlo_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
mlo_val_loader   = DataLoader(mlo_val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
mlo_test_loader  = DataLoader(mlo_test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

paired_test_loader = DataLoader(paired_test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
num_classes = len(classes)

def ordinal_loss(predictions, targets):
    num_classes = predictions.size(1) + 1
    levels = torch.arange(num_classes - 1).to(predictions.device)
    binary_labels = (targets.view(-1, 1) > levels).float()
    return nn.BCEWithLogitsLoss()(predictions, binary_labels)

def coral_logits_to_label(logits):
    probas = torch.sigmoid(logits)
    return torch.sum(probas > 0.5, dim=1)

def coral_logits_to_proba(logits):
    cum_probs = torch.sigmoid(logits)
    batch_size = logits.size(0)
    k = logits.size(1) + 1

    probs = torch.zeros(batch_size, k, device=logits.device)
    probs[:, 0] = 1 - cum_probs[:, 0]

    for i in range(1, k - 1):
        probs[:, i] = cum_probs[:, i - 1] - cum_probs[:, i]

    probs[:, k - 1] = cum_probs[:, k - 2]
    return probs

In [ ]:
class ViTOrdinal(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224", num_classes=4, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        self.head = nn.Linear(in_features, num_classes - 1)

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.head(feats)
        return logits

In [ ]:
def train_one_epoch(model, loader, optimizer, device, num_classes):
    model.train()
    running_loss = 0.0
    total = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = ordinal_loss(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)

    return running_loss / total

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        logits = model(imgs)
        preds = coral_logits_to_label(logits)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    return acc, mae, all_labels, all_preds

In [ ]:
def adjacent_accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred) <= 1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cc_model = ViTOrdinal(num_classes=num_classes).to(device)
cc_optimizer = torch.optim.AdamW(cc_model.parameters(), lr=3e-5)

best_val_mae = float("inf")
best_cc_path = "birads_dataset/cc_vit_ordinal.pt"

epochs = 20

for epoch in range(epochs):
    train_loss = train_one_epoch(cc_model, cc_train_loader, cc_optimizer, device, num_classes)
    val_acc, val_mae, _, _ = evaluate(cc_model, cc_val_loader, device)

    print(f"[CC] Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_mae={val_mae:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(cc_model.state_dict(), best_cc_path)

print("Best CC model saved to:", best_cc_path)

cuda
[CC] Epoch 1/10 | train_loss=0.4968 | val_acc=0.4528 | val_mae=0.8491
[CC] Epoch 2/10 | train_loss=0.4435 | val_acc=0.4623 | val_mae=0.8396
[CC] Epoch 3/10 | train_loss=0.3971 | val_acc=0.3585 | val_mae=0.8302
[CC] Epoch 4/10 | train_loss=0.3679 | val_acc=0.3208 | val_mae=0.8491
[CC] Epoch 5/10 | train_loss=0.3499 | val_acc=0.3868 | val_mae=0.8491
[CC] Epoch 6/10 | train_loss=0.3293 | val_acc=0.4245 | val_mae=0.8019
[CC] Epoch 7/10 | train_loss=0.2962 | val_acc=0.5094 | val_mae=0.8019
[CC] Epoch 8/10 | train_loss=0.2737 | val_acc=0.4340 | val_mae=0.8585
[CC] Epoch 9/10 | train_loss=0.2597 | val_acc=0.5000 | val_mae=0.7642
[CC] Epoch 10/10 | train_loss=0.2399 | val_acc=0.5566 | val_mae=0.6509
Best CC model saved to: birads_dataset/cc_vit_ordinal.pt


In [ ]:
from collections import Counter

cc_model.load_state_dict(torch.load(best_cc_path, map_location=device))

val_acc, val_mae, y_true, y_pred = evaluate(cc_model, cc_val_loader, device)

print("Val true counts:", Counter(y_true))
print("Val pred counts:", Counter(y_pred))

f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")
adj_acc = adjacent_accuracy(y_true, y_pred)
qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

print("Val accuracy:", val_acc)
print("Val MAE:", val_mae)
print("Val F1 macro:", f1_macro)
print("Val F1 weighted:", f1_weighted)
print("Val adjacent accuracy:", adj_acc)
print("Val QWK:", qwk)
print("Val true counts:", Counter(y_true))
print("Val pred counts:", Counter(y_pred))

C:\Users\Roofu\AppData\Local\Temp\ipykernel_20520\1274294857.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cc_model.load_state_dict(torch.load(best_cc_path, map_locati

Val true counts: Counter({np.int64(3): 51, np.int64(4): 18, np.int64(1): 14, np.int64(2): 14, np.int64(0): 9})
Val pred counts: Counter({np.int64(3): 64, np.int64(1): 18, np.int64(2): 12, np.int64(0): 8, np.int64(4): 4})
Val accuracy: 0.5566037735849056
Val MAE: 0.6509433962264151
Val F1 macro: 0.4664985334358736
Val F1 weighted: 0.5273611522145759
Val adjacent accuracy: 0.8490566037735849
Val QWK: 0.5105590951392369
Val true counts: Counter({np.int64(3): 51, np.int64(4): 18, np.int64(1): 14, np.int64(2): 14, np.int64(0): 9})
Val pred counts: Counter({np.int64(3): 64, np.int64(1): 18, np.int64(2): 12, np.int64(0): 8, np.int64(4): 4})


In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)
cm_df

,0,1,2,3,4
0,5,1,0,2,1
1,0,9,2,3,0
2,2,2,3,7,0
3,0,6,6,39,0
4,1,0,1,13,3


In [ ]:
mlo_model = ViTOrdinal(num_classes=num_classes).to(device)
mlo_optimizer = torch.optim.AdamW(mlo_model.parameters(), lr=3e-5)

best_val_mae = float("inf")
best_mlo_path = "birads_dataset/mlo_vit_ordinal.pt"

epochs = 15

for epoch in range(epochs):
    train_loss = train_one_epoch(mlo_model, mlo_train_loader, mlo_optimizer, device, num_classes)
    val_acc, val_mae, _, _ = evaluate(mlo_model, mlo_val_loader, device)

    print(f"[MLO] Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_mae={val_mae:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(mlo_model.state_dict(), best_mlo_path)

print("Best MLO model saved to:", best_mlo_path)

[MLO] Epoch 1/10 | train_loss=0.4872 | val_acc=0.4340 | val_mae=0.7830
[MLO] Epoch 2/10 | train_loss=0.4136 | val_acc=0.4623 | val_mae=0.7925
[MLO] Epoch 3/10 | train_loss=0.3820 | val_acc=0.4811 | val_mae=0.7547
[MLO] Epoch 4/10 | train_loss=0.3457 | val_acc=0.5094 | val_mae=0.6792
[MLO] Epoch 5/10 | train_loss=0.3368 | val_acc=0.4906 | val_mae=0.7642
[MLO] Epoch 6/10 | train_loss=0.3068 | val_acc=0.5000 | val_mae=0.6604
[MLO] Epoch 7/10 | train_loss=0.2917 | val_acc=0.5094 | val_mae=0.6509
[MLO] Epoch 8/10 | train_loss=0.2736 | val_acc=0.4623 | val_mae=0.7358
[MLO] Epoch 9/10 | train_loss=0.2330 | val_acc=0.4434 | val_mae=0.8396
[MLO] Epoch 10/10 | train_loss=0.2401 | val_acc=0.5189 | val_mae=0.6509
Best MLO model saved to: birads_dataset/mlo_vit_ordinal.pt


In [ ]:
mlo_model.load_state_dict(torch.load(best_mlo_path, map_location=device))

val_acc, val_mae, y_true, y_pred = evaluate(mlo_model, mlo_val_loader, device)

print("Val true counts:", Counter(y_true))
print("Val pred counts:", Counter(y_pred))

f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")
adj_acc = adjacent_accuracy(y_true, y_pred)
qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

print("Val accuracy:", val_acc)
print("Val MAE:", val_mae)
print("Val F1 macro:", f1_macro)
print("Val F1 weighted:", f1_weighted)
print("Val adjacent accuracy:", adj_acc)
print("Val QWK:", qwk)
print("Val true counts:", Counter(y_true))
print("Val pred counts:", Counter(y_pred))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)
cm_df

C:\Users\Roofu\AppData\Local\Temp\ipykernel_20520\2097122894.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mlo_model.load_state_dict(torch.load(best_mlo_path, map_loca

Val true counts: Counter({np.int64(3): 51, np.int64(4): 18, np.int64(1): 14, np.int64(2): 14, np.int64(0): 9})
Val pred counts: Counter({np.int64(3): 66, np.int64(2): 14, np.int64(4): 14, np.int64(0): 8, np.int64(1): 4})
Val accuracy: 0.5094339622641509
Val MAE: 0.6509433962264151
Val F1 macro: 0.43045230912877974
Val F1 weighted: 0.4890773072654316
Val adjacent accuracy: 0.8773584905660378
Val QWK: 0.5633489200623469
Val true counts: Counter({np.int64(3): 51, np.int64(4): 18, np.int64(1): 14, np.int64(2): 14, np.int64(0): 9})
Val pred counts: Counter({np.int64(3): 66, np.int64(2): 14, np.int64(4): 14, np.int64(0): 8, np.int64(1): 4})


,0,1,2,3,4
0,5,0,1,3,0
1,1,3,5,5,0
2,1,0,4,8,1
3,1,1,4,37,8
4,0,0,0,13,5


In [ ]:
cc_model.load_state_dict(torch.load(best_cc_path, map_location=device))
mlo_model.load_state_dict(torch.load(best_mlo_path, map_location=device))

cc_test_acc, cc_test_mae, cc_y_true, cc_y_pred = evaluate(cc_model, cc_test_loader, device)
mlo_test_acc, mlo_test_mae, mlo_y_true, mlo_y_pred = evaluate(mlo_model, mlo_test_loader, device)

print("CC Test Acc:", cc_test_acc)
print("CC Test MAE:", cc_test_mae)

print("MLO Test Acc:", mlo_test_acc)
print("MLO Test MAE:", mlo_test_mae)

C:\Users\Roofu\AppData\Local\Temp\ipykernel_20520\4215052071.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cc_model.load_state_dict(torch.load(best_cc_path, map_locati

CC Test Acc: 0.514018691588785
CC Test MAE: 0.6635514018691588
MLO Test Acc: 0.4672897196261682
MLO Test MAE: 0.7476635514018691


In [ ]:
@torch.no_grad()
def evaluate_ensemble(cc_model, mlo_model, loader, device):
    cc_model.eval()
    mlo_model.eval()

    all_preds = []
    all_labels = []

    for cc_imgs, mlo_imgs, labels in loader:
        cc_imgs = cc_imgs.to(device)
        mlo_imgs = mlo_imgs.to(device)

        cc_logits = cc_model(cc_imgs)
        mlo_logits = mlo_model(mlo_imgs)

        cc_probs = coral_logits_to_proba(cc_logits)
        mlo_probs = coral_logits_to_proba(mlo_logits)

        avg_probs = (cc_probs + mlo_probs) / 2.0
        preds = torch.argmax(avg_probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    return acc, mae, all_labels, all_preds

In [ ]:
ens_acc, ens_mae, ens_y_true, ens_y_pred = evaluate_ensemble(
    cc_model, mlo_model, paired_test_loader, device
)

print("Ensemble Test Acc:", ens_acc)
print("Ensemble Test MAE:", ens_mae)

Ensemble Test Acc: 0.6074766355140186
Ensemble Test MAE: 0.514018691588785


In [ ]:
ens_y_true_orig = [inv_label_map[y] for y in ens_y_true]
ens_y_pred_orig = [inv_label_map[y] for y in ens_y_pred]

print("First 20 true labels:", ens_y_true_orig[:20])
print("First 20 pred labels:", ens_y_pred_orig[:20])

First 20 true labels: [np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(0), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(0), np.int64(2), np.int64(4)]
First 20 pred labels: [np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(0), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(2), np.int64(0), np.int64(2), np.int64(4)]
